# Cayley/RoBERTa Colab Runtime Setup

This notebook runs on a Colab GPU kernel while loading your project files from Drive, GitHub, or an uploaded zip.

## 1. Runtime check

In Colab, use `Runtime > Change runtime type > GPU` before running the rest of the notebook.

In [ ]:
import os
import platform
import sys

print('Python:', sys.version)
print('Platform:', platform.platform())
!nvidia-smi || true

## 2. Choose project source

Use `drive` if your project folder is in Google Drive. Use `github` if the repo is pushed. Use `upload` for a zipped copy of the project.

In [ ]:
PROJECT_SOURCE = 'github'  # 'drive', 'github', or 'upload'

# Drive settings
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/cayley'

# GitHub settings. Leave GITHUB_TOKEN empty for public repos.
# For private repos, add a Colab Secret named GITHUB_TOKEN with repo read access.
GITHUB_REPO = 'https://github.com/picramide/cayley.git'
GITHUB_BRANCH = 'main'
GITHUB_TOKEN = ''
GITHUB_TOKEN_SECRET = 'GITHUB_TOKEN'

RUNTIME_PROJECT_DIR = '/content/cayley'
REQUIREMENTS_FILE = 'requirements-colab.txt'

## 3. Make project files available to Colab

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess


def run(cmd, cwd=None):
    print('+', cmd, flush=True)
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    completed = subprocess.run(cmd, shell=True, check=False, cwd=cwd, env=env)
    if completed.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {completed.returncode}: {cmd}')
    return completed

if PROJECT_SOURCE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
    project_dir = Path(DRIVE_PROJECT_DIR)
    if not project_dir.exists():
        raise FileNotFoundError(f'Drive project folder not found: {project_dir}')

elif PROJECT_SOURCE == 'github':
    repo_url = GITHUB_REPO
    github_token = GITHUB_TOKEN
    if not github_token:
        try:
            from google.colab import userdata
            github_token = userdata.get(GITHUB_TOKEN_SECRET) or ''
        except Exception:
            github_token = ''
    if github_token:
        repo_url = repo_url.replace('https://', f'https://x-access-token:{github_token}@')
    elif 'github.com' in repo_url:
        print('No GitHub token found. Public repos can clone without one; private repos need a Colab Secret named GITHUB_TOKEN.')
    project_dir = Path(RUNTIME_PROJECT_DIR)
    if project_dir.exists():
        shutil.rmtree(project_dir)
    run(f'git clone --branch {GITHUB_BRANCH} --depth 1 {repo_url} {project_dir}')

elif PROJECT_SOURCE == 'upload':
    from google.colab import files
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.endswith('.zip')]
    if not zip_names:
        raise ValueError('Upload a .zip file containing the project.')
    project_dir = Path(RUNTIME_PROJECT_DIR)
    if project_dir.exists():
        shutil.rmtree(project_dir)
    project_dir.mkdir(parents=True)
    run(f'unzip -q {zip_names[0]} -d {project_dir}')
    children = [p for p in project_dir.iterdir() if p.is_dir()]
    if len(children) == 1 and not (project_dir / REQUIREMENTS_FILE).exists():
        project_dir = children[0]

else:
    raise ValueError(f'Unknown PROJECT_SOURCE: {PROJECT_SOURCE}')

print('Project dir:', project_dir)
print('Top-level files:', sorted(p.name for p in project_dir.iterdir())[:30])


## 4. Install dependencies and project

In [ ]:
import sys
from pathlib import Path

req_path = Path(project_dir) / REQUIREMENTS_FILE
if req_path.exists():
    run(f'{sys.executable} -m pip install -q -r {req_path}')
else:
    run(f'{sys.executable} -m pip install -q torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu121')
    run(f'{sys.executable} -m pip install -q transformers datasets evaluate accelerate scikit-learn networkx pandas tqdm matplotlib seaborn einops wandb')

if (Path(project_dir) / 'pyproject.toml').exists() or (Path(project_dir) / 'setup.py').exists():
    run(f'{sys.executable} -m pip install -q -e {project_dir}')

if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

print('sys.path[0]:', sys.path[0])

## 5. Verify PyTorch, Transformers, and RoBERTa

In [ ]:
import torch
import transformers
from transformers import AutoConfig, AutoModel, AutoTokenizer

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
print('transformers:', transformers.__version__)

model_name = 'roberta-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)
config = AutoConfig.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name, config=config)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

inputs = tokenizer('cayley graph transformer pattern benchmark smoke test', return_tensors='pt').to(device)
with torch.no_grad():
    outputs = model(**inputs)
print('last_hidden_state:', tuple(outputs.last_hidden_state.shape))

## 6. Run MRPC dense and window benchmarks

This calls the general benchmark entrypoint twice: MRPC on dense attention, then MRPC on a generated local bidirectional window mask. Training uses full MRPC train/validation splits and shows live progress in the notebook output.


In [ ]:
AUTO_DOWNLOAD_RESULTS = True
DENSE_RESULTS_FILE = 'results/mrpc_dense.jsonl'
WINDOW_RESULTS_FILE = 'results/mrpc_local_w16.jsonl'
WINDOW_MASK_PATH = 'masks/local_w16_128.pt'

run(
    'python -u scripts/generate_masks.py '
    '--kind local '
    '--seq 128 '
    '--window 16 '
    f'--output {WINDOW_MASK_PATH}',
    cwd=project_dir,
)

BENCHMARK_BASE = (
    'python -u scripts/benchmark_roberta_glue.py '
    '--task_name mrpc '
    '--dataset_name nyu-mll/glue '
    '--model_name FacebookAI/roberta-base '
    '--do_train '
    '--do_eval '
    '--max_length 128 '
    '--num_train_epochs 5.0 '
    '--seed 42 '
    '--per_device_train_batch_size 8 '
    '--per_device_eval_batch_size 16 '
    '--learning_rate 2e-5 '
)

run(
    BENCHMARK_BASE +
    '--mask_name dense '
    '--output_dir outputs/mrpc_dense '
    f'--results_file {DENSE_RESULTS_FILE} '
    '--run_name mrpc_dense',
    cwd=project_dir,
)

run(
    BENCHMARK_BASE +
    '--mask_name local_w16 '
    f'--mask_path {WINDOW_MASK_PATH} '
    '--output_dir outputs/mrpc_local_w16 '
    f'--results_file {WINDOW_RESULTS_FILE} '
    '--run_name mrpc_local_w16',
    cwd=project_dir,
)

result_files = [project_dir / DENSE_RESULTS_FILE, project_dir / WINDOW_RESULTS_FILE]
for result_path in result_files:
    if not result_path.exists():
        raise FileNotFoundError(f'Expected results file not found: {result_path}')
    print(f'Results saved at: {result_path}')
    print(result_path.read_text())

if AUTO_DOWNLOAD_RESULTS:
    try:
        from google.colab import files
        for result_path in result_files:
            files.download(str(result_path))
    except Exception as exc:
        print(f'Automatic browser download failed: {exc}')


In [ ]:
AUTO_DOWNLOAD_RESULTS = True
DENSE_RESULTS_FILE = 'results/mrpc_dense.jsonl'
WINDOW_RESULTS_FILE = 'results/mrpc_local_w16.jsonl'
WINDOW_MASK_PATH = 'masks/local_w16_128.pt'

run(
    'python -u scripts/generate_masks.py '
    '--kind local '
    '--seq 128 '
    '--window 16 '
    f'--output {WINDOW_MASK_PATH}',
    cwd=project_dir,
)

BENCHMARK_BASE = (
    'python -u scripts/benchmark_roberta_glue.py '
    '--task_name mrpc '
    '--dataset_name nyu-mll/glue '
    '--model_name FacebookAI/roberta-base '
    '--do_train '
    '--do_eval '
    '--max_length 128 '
    '--num_train_epochs 5.0 '
    '--seed 42 '
    '--per_device_train_batch_size 8 '
    '--per_device_eval_batch_size 16 '
    '--learning_rate 2e-5 '
)

run(
    BENCHMARK_BASE +
    '--mask_name dense '
    '--output_dir outputs/mrpc_dense '
    f'--results_file {DENSE_RESULTS_FILE} '
    '--run_name mrpc_dense',
    cwd=project_dir,
)

run(
    BENCHMARK_BASE +
    '--mask_name local_w16 '
    f'--mask_path {WINDOW_MASK_PATH} '
    '--output_dir outputs/mrpc_local_w16 '
    f'--results_file {WINDOW_RESULTS_FILE} '
    '--run_name mrpc_local_w16',
    cwd=project_dir,
)

# Bipartite Cayley mask for MRPC
BIPARTITE_RESULTS_FILE = 'results/mrpc_bipartite.jsonl'
BIPARTITE_MASK_PATH = 'masks/bipartite_128_64.pt'

run(
    'python -u scripts/generate_masks.py '
    '--kind bipartite '
    '--seq 128 '
    '--premise_len 64 '
    '--local_window 3 '
    '--cross_window 2 '
    f'--output {BIPARTITE_MASK_PATH}',
    cwd=project_dir,
)

run(
    BENCHMARK_BASE +
    '--mask_name bipartite '
    f'--mask_path {BIPARTITE_MASK_PATH} '
    '--output_dir outputs/mrpc_bipartite '
    f'--results_file {BIPARTITE_RESULTS_FILE} '
    '--run_name mrpc_bipartite',
    cwd=project_dir,
)

result_files = [project_dir / DENSE_RESULTS_FILE, project_dir / WINDOW_RESULTS_FILE, project_dir / BIPARTITE_RESULTS_FILE]
for result_path in result_files:
    if not result_path.exists():
        raise FileNotFoundError(f'Expected results file not found: {result_path}')
    print(f'Results saved at: {result_path}')
    print(result_path.read_text())

if AUTO_DOWNLOAD_RESULTS:
    try:
        from google.colab import files
        for result_path in result_files:
            files.download(str(result_path))
    except Exception as exc:
        print(f'Automatic browser download failed: {exc}')

In [37]:
print(open("/content/cayley/results/mrpc_dense.jsonl").read())
print(open("/content/cayley/results/mrpc_local_w16.jsonl").read())

{"dataset_name": "nyu-mll/glue", "learning_rate": 2e-05, "mask_name": "dense", "mask_path": null, "max_length": 128, "metrics": {"epoch": 5.0, "eval_accuracy": 0.8995098039215687, "eval_f1": 0.9266547406082289, "eval_loss": 0.563569962978363, "eval_runtime": 0.6255, "eval_samples_per_second": 652.239, "eval_steps_per_second": 41.564}, "model_name": "FacebookAI/roberta-base", "num_train_epochs": 5.0, "per_device_eval_batch_size": 16, "per_device_train_batch_size": 8, "seed": 42, "task_name": "mrpc"}

{"dataset_name": "nyu-mll/glue", "learning_rate": 2e-05, "mask_name": "local_w16", "mask_path": "masks/local_w16_128.pt", "max_length": 128, "metrics": {"epoch": 5.0, "eval_accuracy": 0.6887254901960784, "eval_f1": 0.7851099830795262, "eval_loss": 1.3470855951309204, "eval_runtime": 0.9825, "eval_samples_per_second": 415.254, "eval_steps_per_second": 26.462}, "model_name": "FacebookAI/roberta-base", "num_train_epochs": 5.0, "per_device_eval_batch_size": 16, "per_device_train_batch_size": 8,